In [ ]:
# ── CELL 1: Install ──────────────────────────────────────
!pip install -U transformers librosa soundfile deep-translator

In [ ]:
# ── CELL 2: Load ASR model ───────────────────────────────
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
import torch, librosa

ASR_MODEL_ID  = "Lingalingeswaran/whisper-small-sinhala"
asr_processor = AutoProcessor.from_pretrained(ASR_MODEL_ID)
asr_model     = AutoModelForSpeechSeq2Seq.from_pretrained(ASR_MODEL_ID)

device    = "cuda" if torch.cuda.is_available() else "cpu"
asr_model = asr_model.to(device)

asr_model.generation_config.language = "sinhala"
asr_model.generation_config.task     = "transcribe"
print("✅ ASR model loaded")

In [ ]:
# ── CELL 3: Google Translate function ────────────────────
from deep_translator import GoogleTranslator

def translate_sinhala_to_english(sinhala_text):
    try:
        translated = GoogleTranslator(source="si", target="en").translate(sinhala_text)
        return translated
    except Exception as e:
        return f"[Translation error: {e}]"

In [ ]:
# ── CELL 4: Transcribe function ──────────────────────────
def transcribe_sinhala(audio_path):
    audio_array, _ = librosa.load(audio_path, sr=16000)
    input_features = asr_processor(
        audio_array, sampling_rate=16000, return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(input_features)

    return asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

In [ ]:
# ── CELL 5: Full pipeline ────────────────────────────────
def sinhala_voice_to_english(audio_path):
    print(f"\n🎙️  Processing: {audio_path}")
    sinhala = transcribe_sinhala(audio_path)
    print(f"   Sinhala transcript : {sinhala}")
    english = translate_sinhala_to_english(sinhala)
    print(f"   English translation: {english}")
    return {"sinhala_text": sinhala, "english_text": english}

In [ ]:
result = sinhala_voice_to_english("famer-1.ogg")

In [ ]:
# ── Load SVM classifier ──────────────────────────────────
import joblib, re

classifier = joblib.load("paddyguard_best_classifier.pkl")
tfidf      = joblib.load("paddyguard_tfidf.pkl")

LABEL_MAP = {
    0: "Bacterial Blight",
    1: "Leaf Blast",
    2: "Brown Spot",
    3: "Healthy"
}

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def classify_disease(english_text, threshold=0.70):
    clean      = preprocess_text(english_text)
    vec        = tfidf.transform([clean])
    proba      = classifier.predict_proba(vec)[0]
    label_id   = int(proba.argmax())
    confidence = float(proba.max())
    return {
        "disease"        : LABEL_MAP[label_id],
        "label_id"       : label_id,
        "confidence"     : round(confidence, 3),
        "needs_followup" : confidence < threshold,
        "all_scores"     : {LABEL_MAP[i]: round(float(p),3) for i,p in enumerate(proba)}
    }

print("✅ SVM classifier loaded")

In [ ]:
# ── Full end-to-end pipeline ─────────────────────────────
def full_pipeline(audio_path):
    print(f"\n🎙️  File: {audio_path}")
    print("─" * 45)

    # Step 1: Voice → Sinhala → English
    result = sinhala_voice_to_english(audio_path)

    # Step 2: English → Disease classification
    diagnosis = classify_disease(result["english_text"])

    print(f"🌾  Disease    : {diagnosis['disease']}")
    print(f"📊  Confidence : {diagnosis['confidence']*100:.1f}%")
    print(f"❓  Follow-up  : {'YES — ask more' if diagnosis['needs_followup'] else 'NO — confident'}")
    print(f"📈  All scores : {diagnosis['all_scores']}")
    return {**result, **diagnosis}

# Test
result = full_pipeline("famer-1.ogg")

In [ ]:
result = full_pipeline("famer-3.ogg")

In [ ]:
result = full_pipeline("famer-4.ogg")